In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ==========================================
# RAG PDF Assistant — Qwen2.5-7B + Gradio
# ==========================================

!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate gradio

import os
import torch
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import faiss
import numpy as np

# --- ۱. پیدا کردن فایل (PDF یا TXT) ---
FILE_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith((".pdf", ".txt")):
            FILE_PATH = os.path.join(root, f)
            break
    if FILE_PATH:
        break

if not FILE_PATH:
    raise Exception("❌ هیچ فایل PDF یا TXT پیدا نشد.")

print("✅ فایل:", FILE_PATH)

# --- ۲. خواندن متن ---
if FILE_PATH.lower().endswith(".pdf"):
    reader = PdfReader(FILE_PATH)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if text and text.strip():
            pages.append({"page": i, "text": text.strip()})
else:
    with open(FILE_PATH, "r", encoding="utf-8") as f:
        full_text = f.read()
    paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]
    pages = [{"page": i+1, "text": p} for i, p in enumerate(paragraphs)]

print(f"تعداد بخش‌ها: {len(pages)}")

# --- ۳. تقسیم به قطعه ---
def split_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = []
for p in pages:
    for chunk_text in split_text(p["text"]):
        chunks.append({
            "text": chunk_text,
            "source": os.path.basename(FILE_PATH),
            "page": p["page"]
        })

print(f"تعداد قطعه‌ها: {len(chunks)}")

# --- ۴. Embedding + FAISS ---
print("\nدر حال ساخت Embedding...")
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
chunk_texts = [c["text"] for c in chunks]
embeddings = embedder.encode(chunk_texts, convert_to_numpy=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"تعداد بردار در FAISS: {index.ntotal}")

# --- ۵. مدل ۷B ---
print("\nدر حال بارگذاری Qwen2.5-7B-Instruct (چند دقیقه صبر کن)...")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("مدل بارگذاری شد ✅")

# --- ۶. تابع RAG ---
def rag_query(question, top_k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb, top_k)
    retrieved = [chunks[i] for i in indices[0]]
    context = "\n\n".join([
        f"[منبع: {c['source']}، صفحه/پاراگراف {c['page']}]\n{c['text']}"
        for c in retrieved
    ])

    messages = [
        {"role": "system", "content": "تو یک دستیار هستی که فقط بر اساس متن داده‌شده جواب می‌دهد. اگر جواب در متن نبود، دقیقاً بگو «در متن نیست». همیشه فارسی جواب بده و در پایان، منبع را ذکر کن."},
        {"role": "user", "content": f"متن:\n{context}\n\nسوال: {question}\n\nجواب:"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.1,
        )

    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sources = list({(c["source"], c["page"]) for c in retrieved})

    return {"answer": answer.strip(), "sources": sources}

# --- ۷. رابط Gradio ---
def chat_fn(message, history):
    if not message.strip():
        return "لطفاً یک سوال بنویس."
    result = rag_query(message)
    answer = result["answer"]
    if "در متن نیست" in answer:
        return f"❌ {answer}"
    sources_text = "\n".join([f"📄 {src}، بخش {p}" for src, p in result["sources"]])
    return f"{answer}\n\n---\n**منابع:**\n{sources_text}"

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🤖 دستیار RAG — پرسش از سند",
    description="سوال خودت را بپرس. اگر جواب در سند نبود، می‌گویم «در متن نیست».",
    examples=[
        "Transformer چیست؟",
        "تفاوت BERT و GPT چیست؟",
        "پایتخت فرانسه کجاست؟",
    ],
)

demo.launch(share=True)